In [1]:
import numpy as np
import scipy.optimize as sco
import pandas as pd
from scipy.optimize import minimize
import statsmodels.api as sm
import statsmodels.stats.api as sms
import matplotlib.pyplot as plt
import yfinance as yf
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.float_format', lambda x: '{:.2f}'.format(x))

In [2]:
data = pd.read_csv("data_ind_precios.csv")
data = data.interpolate(limit_direction="both")
data.set_index("Date", inplace = True)
data.drop(columns = "FB", inplace = True)
data #Los datos ya estan en terminos mensuales

,AAPL,BA,CAT,CSCO,CVX,HD,JNJ,JPM,MCD,MSFT,NKE,PG,TRV,V
Date,,,,,,,,,,,,,,
2015-07-01,27.17,127.69,60.70,20.68,57.54,92.47,75.87,51.83,77.89,40.49,51.04,58.07,84.93,70.26
2015-08-01,25.25,115.74,59.55,18.97,52.67,92.02,71.15,48.79,74.12,37.74,49.51,53.94,79.67,66.50
2015-09-01,24.82,116.72,50.92,19.25,51.94,91.25,71.22,46.41,77.53,38.63,54.48,54.91,79.65,65.07
2015-10-01,26.89,131.98,56.87,21.15,59.84,98.19,77.07,48.91,88.33,45.94,58.19,58.30,90.91,72.47
2015-11-01,26.62,129.64,57.23,20.14,60.13,106.32,77.23,51.12,89.83,47.44,58.75,57.63,92.26,73.81
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-01,241.26,174.63,342.30,63.30,154.99,391.67,162.32,261.77,304.83,395.47,78.51,172.76,256.38,361.50
2025-03-01,221.84,170.55,328.22,60.93,165.28,361.93,164.43,242.63,308.83,374.70,62.74,169.36,262.30,349.88
2025-04-01,212.22,183.24,307.79,57.00,134.42,358.26,154.98,241.96,317.85,394.54,56.03,161.56,263.05,344.93


In [3]:
log_retornos_mensuales = np.log(data / data.shift()).dropna()
log_retornos_anualizados = log_retornos_mensuales.mean() * 12
log_retornos_anualizados

AAPL   0.20
BA     0.05
CAT    0.19
CSCO   0.12
CVX    0.09
HD     0.14
JNJ    0.07
JPM    0.17
MCD    0.13
MSFT   0.25
NKE    0.03
PG     0.10
TRV    0.12
V      0.16
dtype: float64

In [4]:
corr = log_retornos_mensuales.corr().sort_values(by = "MSFT", ascending = False)
mas_corr = corr[1: 6].index.to_list()

In [5]:
mas_corr

['AAPL', 'V', 'HD', 'NKE', 'CSCO']

In [6]:
datos = data[mas_corr]
datos = np.log(datos / datos.shift()).dropna()
datos

,AAPL,V,HD,NKE,CSCO
Date,,,,,
2015-08-01,-0.07,-0.06,-0.00,-0.03,-0.09
2015-09-01,-0.02,-0.02,-0.01,0.10,0.01
2015-10-01,0.08,0.11,0.07,0.07,0.09
2015-11-01,-0.01,0.02,0.08,0.01,-0.05
2015-12-01,-0.11,-0.02,-0.01,-0.06,-0.00
...,...,...,...,...,...
2025-02-01,0.02,0.06,-0.04,0.03,0.06
2025-03-01,-0.08,-0.03,-0.08,-0.22,-0.04
2025-04-01,-0.04,-0.01,-0.01,-0.11,-0.07


In [7]:
def port_ret(weights):
    return np.sum(datos.mean() * weights) * 12

def port_vol(weights):
    return np.sqrt(np.dot(weights.T, np.dot(datos.cov() * 12, weights)))

def max_sharpe(weights):
    return - port_ret(weights) / port_vol(weights)


In [8]:
noa = len(mas_corr)
cons = ({"type": "eq", "fun" : lambda x: np.sum(x) - 1}, {'type': "ineq", 'fun': lambda x: sum([-i if i < 0.04 else 0 for i in x])}
       )
pesos_iniciales = np.array([1 / noa for x in range(noa)])
bounds = tuple((0,0.32) for _ in range (noa)) 

maximizar = sco.minimize(max_sharpe, pesos_iniciales, bounds= bounds, constraints=cons, method='SLSQP')
maximizar

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: -0.9168952354889189
       x: [ 3.200e-01  3.200e-01  2.744e-01  3.278e-16  8.555e-02]
     nit: 3
     jac: [ 4.087e-02 -9.946e-02  5.166e-02  6.053e-01  5.339e-02]
    nfev: 19
    njev: 3

In [9]:
print(maximizar.x)

[3.20000000e-01 3.20000000e-01 2.74448502e-01 3.27793928e-16
 8.55514975e-02]


In [10]:
retornos_del_portafolio = port_ret(maximizar.x)
retornos_del_portafolio

0.16586189759701458

In [11]:
volatilidad_del_portafolio = port_vol(maximizar.x)
volatilidad_del_portafolio

0.1808951461161989

In [12]:
ratio_de_sharpe = retornos_del_portafolio / volatilidad_del_portafolio
ratio_de_sharpe

0.9168952354889189

In [13]:
dicc = {}
for x , y in zip(mas_corr,maximizar.x):
    dicc.update({x:y})
dicc

{'AAPL': 0.32,
 'V': 0.3199999999999996,
 'HD': 0.27444850248592734,
 'NKE': 3.2779392781600583e-16,
 'CSCO': 0.08555149751407275}

# EPH

In [36]:
df = pd.read_excel("usu_individual_T224.xlsx")
df

,CODUSU,ANO4,TRIMESTRE,NRO_HOGAR,COMPONENTE,H15,REGION,MAS_500,AGLOMERADO,PONDERA,...,PDECIFR,ADECIFR,IPCF,DECCFR,IDECCFR,RDECCFR,GDECCFR,PDECCFR,ADECCFR,PONDIH
0,TQRMNORUUHJLMRCDEIIAD00861800,2024,2,1,2,1,1,S,32,2678,...,NaN,6,500000.00,9,NaN,8,8.00,NaN,6,4604
1,TQRMNORPRHKLMUCDEIIAD00858515,2024,2,1,1,1,1,S,32,2912,...,NaN,7,455000.00,8,NaN,8,8.00,NaN,6,3713
2,TQRMNORPRHKLMUCDEIIAD00858515,2024,2,1,2,1,1,S,32,2912,...,NaN,7,455000.00,8,NaN,8,8.00,NaN,6,3713
3,TQRMNORPRHKLMUCDEIIAD00858515,2024,2,1,3,1,1,S,32,2912,...,NaN,7,455000.00,8,NaN,8,8.00,NaN,6,3713
4,TQRMNOSSUHKLMUCDEIIAD00858728,2024,2,1,1,1,1,S,32,1519,...,NaN,12,0.00,12,NaN,12,12.00,NaN,12,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47146,TQRMNORQQHJLLMCDEGKDB00866867,2024,2,1,3,0,43,N,14,152,...,9.00,9,275000.00,6,6.00,5,NaN,7.00,7,489
47147,TQRMNORQQHJLLMCDEGKDB00866867,2024,2,1,4,0,43,N,14,152,...,9.00,9,275000.00,6,6.00,5,NaN,7.00,7,489
47148,TQRMNORRTHJKLNCDEGKDB00866868,2024,2,1,1,1,43,N,14,174,...,3.00,3,375000.00,7,8.00,7,NaN,8.00,9,283
47149,TQSMNOPYSHMOKMCDEFMDB00866869,2024,2,1,1,1,43,N,6,183,...,4.00,2,135000.00,3,3.00,2,NaN,3.00,2,272


In [52]:
PEA_GBA = df.loc[df['ESTADO'].isin([1, 2]) & (df['REGION'] == 1), 'PONDERA'].sum()
PEA_GBA

7901666

In [68]:
TD_GBA = df.loc[df["REGION"] == 1].groupby('ESTADO')['PONDERA'].sum()[2] / PEA_GBA
print(f"La Tasa de desocupados en GBA fue de {round(TD_GBA * 100,2)} %")

La Tasa de desocupados en GBA fue de 8.31 %


In [70]:
PEA_PAMP= df.loc[df['ESTADO'].isin([1, 2]) & (df['REGION'] == 43), 'PONDERA'].sum()

TD_PAMP = df.loc[df["REGION"] == 43].groupby('ESTADO')['PONDERA'].sum()[2] / PEA_PAMP
print(f"Tasa de desocupados en la region Pampeana fue de {round(TD_PAMP * 100,2)} %")

Tasa de desocupados en la region Pampeana fue de 7.64 %


In [74]:
PEA_CUYO= df.loc[df['ESTADO'].isin([1, 2]) & (df['REGION'] == 42), 'PONDERA'].sum()

TD_CUYO = df.loc[df["REGION"] == 42].groupby('ESTADO')['PONDERA'].sum()[2] / PEA_CUYO
print(f"Tasa de desocupados en la region Pampeana fue de {round(TD_CUYO * 100,2)} %")

Tasa de desocupados en la region Pampeana fue de 5.12 %


In [78]:
guardado = df.to_csv()

# EJERCICIO 3

In [93]:
objetivo = -np.array([60,140])

A_ub = np.array([[1,2], [-2,-1]])
b_ub = np.array([100,-100])

bounds = np.array([(0,None) for _ in range(2)])

resultado = sco.linprog(objetivo, A_ub= A_ub, b_ub=b_ub, bounds=bounds)
resultado

        message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
        success: True
         status: 0
            fun: -6666.666666666666
              x: [ 3.333e+01  3.333e+01]
            nit: 0
          lower:  residual: [ 3.333e+01  3.333e+01]
                 marginals: [ 0.000e+00  0.000e+00]
          upper:  residual: [       inf        inf]
                 marginals: [ 0.000e+00  0.000e+00]
          eqlin:  residual: []
                 marginals: []
        ineqlin:  residual: [ 0.000e+00  0.000e+00]
                 marginals: [-7.333e+01 -6.667e+00]
 mip_node_count: 0
 mip_dual_bound: 0.0
        mip_gap: 0.0

In [95]:
beneficio = -resultado.fun
print(beneficio)

6666.666666666666


In [97]:
cantidades = resultado.x
print(cantidades)

[33.33333333 33.33333333]


# MACHINE LEARNING

In [100]:
df_ml = df[['CH08','CH04', 'CH06', 'NIVEL_ED', 'CH07', 'ESTADO']]
df_ml

,CH08,CH04,CH06,NIVEL_ED,CH07,ESTADO
0,1,2,66,6,2,3
1,1,2,52,6,3,1
2,12,2,19,5,5,1
3,2,2,14,3,5,3
4,1,2,49,6,1,1
...,...,...,...,...,...,...
47146,1,1,3,7,5,4
47147,1,2,7,1,5,4
47148,1,2,85,2,4,3
47149,4,2,48,3,4,1


In [102]:
df_ml['gen'] = np.where(df_ml['CH04'] == 2, 1, 0)
df_ml['univ'] = np.where(df_ml['NIVEL_ED'] == 6, 1, 0)
df_ml['casado'] = np.where(df_ml['CH07'] == 2, 1, 0)
df_ml['estado'] =[1 if x == 1 else 0 for x in df_ml["ESTADO"]]
df_ml['sin_cobertura'] = [1 if x == 4 or x == 9 else 0 for x in df_ml["CH08"]]

In [122]:
datos_importantes = df_ml[['CH04', 'CH06', 'NIVEL_ED', 'CH07', 'ESTADO', "sin_cobertura"]]
datos_importantes

,CH04,CH06,NIVEL_ED,CH07,ESTADO,sin_cobertura
0,2,66,6,2,3,0
1,2,52,6,3,1,0
2,2,19,5,5,1,0
3,2,14,3,5,3,0
4,2,49,6,1,1,0
...,...,...,...,...,...,...
47146,1,3,7,5,4,0
47147,2,7,1,5,4,0
47148,2,85,2,4,3,0
47149,2,48,3,4,1,1


In [124]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(datos_importantes, test_size=0.3, random_state=40)

In [126]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)

tree_model.fit(df_train[['CH04', 'CH06', 'NIVEL_ED', 'CH07', 'ESTADO']], df_train['sin_cobertura']) 

DecisionTreeClassifier(max_depth=4, random_state=42)

In [128]:
df_test['tree_pred'] = tree_model.predict(df_test[['CH04', 'CH06', 'NIVEL_ED', 'CH07', 'ESTADO']])

df_test['tree_prob'] = tree_model.predict_proba(df_test[['CH04', 'CH06', 'NIVEL_ED', 'CH07', 'ESTADO']])[:, 1]

In [130]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

tree_acc = accuracy_score(df_test['sin_cobertura'], df_test['tree_pred'])
print(f"Precisión árbol de decisión: {tree_acc:.2%}")
print("Matriz de confusión:\n", confusion_matrix(df_test['sin_cobertura'], df_test['tree_pred']))
print("\nReporte de clasificación:\n", classification_report(df_test['sin_cobertura'], df_test['tree_pred']))

Precisión árbol de decisión: 70.52%
Matriz de confusión:
 [[8617 1090]
 [3080 1359]]

Reporte de clasificación:
               precision    recall  f1-score   support

           0       0.74      0.89      0.81      9707
           1       0.55      0.31      0.39      4439

    accuracy                           0.71     14146
   macro avg       0.65      0.60      0.60     14146
weighted avg       0.68      0.71      0.68     14146



## RESULTADOS

Mi Arbol tiene un 74% de precision en los individuos sin prepaga aunque tenga un recall de 89%

y tambien tiene un 55% de precision en los individuos que tienen prepaga con un recall bajo de 31%